In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os

# ノートブックから見て、def_list フォルダへパスを通す
sys.path.append(os.path.abspath('../def_list'))

from run_classification_audition import run_classification_audition

# results = run_classification_audition(X, y)

In [ ]:
pd.set_option('display.max_columns', None) # データフレームの全列を表示する設定
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head(3)

In [ ]:
X = df.drop(columns=['customerID', 'Churn']) # ヒント（説明変数）
y = df['Churn'].map({'Yes': 1, 'No': 0}) # 答え（目的変数）を0/1に変換
# こちらの方が実務でよく見る「文字列の列名リスト」を自動で作る技
obj_cols = X.select_dtypes(include=['object']).columns
X[obj_cols] = X[obj_cols].astype('category')
# drop_first=True をつけるのが、実務的なスマートな書き方
X = pd.get_dummies(X, drop_first=True)
results = run_classification_audition(X, y)

In [ ]:
# 1. 3つのモデルの結果が入った箱（results）から、それぞれのAIを取り出す
for name, pipe in results.items():
    print(f"\n---  {name} の重要度を調査 ---")
    
    # 決定木やランダムフォレストは 'dt' や 'rf' という名前でパイプラインに入れています
    # モデルの本体を取り出すための辞書を用意
    model_name = list(pipe.named_steps.keys())[1] # パイプラインの2番目（[0]=scaler, [1]=model）
    model = pipe.named_steps[model_name]
    
    # 【ロジスティック回帰の場合】（係数を取得）
    if name == 'LogisticRegression':
        importances = model.coef_[0]
        
    # 【決定木・ランダムフォレストの場合】（重要度を取得）
    else:
        importances = model.feature_importances_
        
    # 表にまとめてグラフ化
    df_imp = pd.DataFrame({'Feature': X.columns, 'Importance': abs(importances)}).sort_values(by='Importance', ascending=False)
    
    # グラフ表示
    plt.figure(figsize=(8, 4))
    sns.barplot(x='Importance', y='Feature', data=df_imp.head(10), palette='magma')
    plt.title(f'Top 10 Features: {name}')
    plt.show()